# Tópicos Avançados em Modelos de Difusão

Este notebook cobre a inicialização de pipelines para modelos avançados, como Latent Consistency Models (LCM), controle com IP-Adapters e arquiteturas baseadas em Transformers (DiT).

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, LCMScheduler
from diffusers import StableDiffusionXLPipeline
from diffusers import PixArtAlphaPipeline
from diffusers.utils import load_image

## 1. Latent Consistency Models (LCM)

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
lcm_lora_id = "latent-consistency/lcm-lora-sdv1-5"

pipe_lcm = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe_lcm.load_lora_weights(lcm_lora_id)
pipe_lcm.scheduler = LCMScheduler.from_config(pipe_lcm.scheduler.config)
pipe_lcm.to("cpu")

prompt_lcm = "a highly detailed cinematic photo of an astronaut on mars"
image_lcm = pipe_lcm(
    prompt=prompt_lcm,
    num_inference_steps=4,
    guidance_scale=1.0,
).images[0]

image_lcm.save("lcm_output.png")

## 2. IP-Adapter (Controle por Imagem)

In [ ]:
pipe_ip = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe_ip.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
pipe_ip.to("cpu")
pipe_ip.set_ip_adapter_scale(0.8)

ref_image = load_image("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ip_adapter_diner.png")
prompt_ip = "a futuristic sci-fi city"

image_ip = pipe_ip(
    prompt=prompt_ip,
    ip_adapter_image=ref_image,
    num_inference_steps=30,
).images[0]

image_ip.save("ip_adapter_output.png")

## 3. Diffusion Transformers (DiT)

In [ ]:
pipe_dit = PixArtAlphaPipeline.from_pretrained("PixArt-alpha/PixArt-XL-2-1024-MS", torch_dtype=torch.float16)
pipe_dit.to("cpu")

prompt_dit = "A small cactus with a happy face in the Sahara desert."
image_dit = pipe_dit(
    prompt=prompt_dit,
    num_inference_steps=20
).images[0]

image_dit.save("dit_output.png")